In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.graph_objects as go
import math

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

def load_dataset(filename):
    BASE_DIR = './data'
    EXTENSION = '.csv'
    TARGET_PATH = f'{BASE_DIR}/{filename}{EXTENSION}'
    try:
        return pd.read_csv(TARGET_PATH)
    except FileNotFoundError:
        return None
    except UnicodeDecodeError:
        return pd.read_csv(TARGET_PATH, encoding='euc-kr')

df = load_dataset('크루즈입항정보')

In [8]:
def nice_tick_step(max_value, target_ticks=5):
    """그래프의 숫자 눈금을 읽기 좋은 간격으로 설정"""
    max_value = max(float(max_value), 1)

    raw_step = max_value / target_ticks
    magnitude = 10 ** math.floor(math.log10(raw_step))
    normalized = raw_step / magnitude

    if normalized <= 1:
        step = 1
    elif normalized <= 2:
        step = 2
    elif normalized <= 5:
        step = 5
    else:
        step = 10

    return max(step * magnitude, 1)

In [16]:
# ============================================================
# 1. 비교 데이터 설정
# ============================================================

periods = ["코로나 이전", "코로나 이후"]

# 코로나 전후 실제 집계 수치
arrival_values = [108, 201]
passenger_values = [233991, 299274]

# 변화율 계산 (%)
arrival_change = (
    (arrival_values[1] - arrival_values[0])
    / arrival_values[0]
    * 100
)

passenger_change = (
    (passenger_values[1] - passenger_values[0])
    / passenger_values[0]
    * 100
)


# 색상 설정
BLUE = "#3485D1"
ORANGE = "#F28E2B"

# 코로나 이전은 파란색, 코로나 이후는 주황색
period_colors = [BLUE, ORANGE]


# ============================================================
# 2. 그래프 1: 코로나 전후 입항·승객 규모 비교
# ============================================================

fig1 = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "입항 건수 비교",
        "승객 수 비교"
    ),
    horizontal_spacing=0.16
)


# 왼쪽: 입항 건수 비교
fig1.add_trace(
    go.Bar(
        x=periods,
        y=arrival_values,
        marker_color=period_colors,

        text=[f"{value:,.0f}" for value in arrival_values],
        textposition="inside",
        insidetextanchor="middle",

        textfont=dict(
            color="white",
            size=15
        ),

        name="입항 건수",

        hovertemplate=(
            "%{x}<br>"
            "입항 건수: %{y:,.0f}건"
            "<extra></extra>"
        )
    ),
    row=1,
    col=1
)


# 오른쪽: 승객 수 비교
fig1.add_trace(
    go.Bar(
        x=periods,
        y=passenger_values,
        marker_color=period_colors,

        text=[f"{value:,.0f}" for value in passenger_values],
        textposition="inside",
        insidetextanchor="middle",

        textfont=dict(
            color="white",
            size=15
        ),

        name="승객 수",

        hovertemplate=(
            "%{x}<br>"
            "승객 수: %{y:,.0f}명"
            "<extra></extra>"
        )
    ),
    row=1,
    col=2
)


# 그래프 1 레이아웃
fig1.update_layout(
    title=dict(
        text="코로나 전후 비교: 입항·승객 규모",
        x=0.5,
        y=0.94,
        xanchor="center",
        yanchor="top",
        font=dict(size=22)
    ),

    height=520,

    margin=dict(
        t=125,
        b=75,
        l=80,
        r=80
    ),

    template="plotly_white",
    showlegend=False,
    bargap=0.4,

    font=dict(
        family="Arial, sans-serif",
        size=13,
        color="#333333"
    )
)


# 왼쪽 Y축: 입항 건수
fig1.update_yaxes(
    title_text="입항 건수",
    tickformat=",",
    rangemode="tozero",
    showgrid=True,
    gridcolor="#E8E8E8",
    row=1,
    col=1
)


# 오른쪽 Y축: 승객 수
fig1.update_yaxes(
    title_text="승객 수(명)",
    tickformat=",",
    rangemode="tozero",
    showgrid=True,
    gridcolor="#E8E8E8",
    row=1,
    col=2
)


fig1.update_xaxes(
    showgrid=False
)

fig1.show()


# ============================================================
# 3. 그래프 2: 코로나 전후 변화율 비교
# ============================================================

metrics = [
    "입항 건수",
    "승객 수"
]

change_rates = [
    arrival_change,
    passenger_change
]

change_text = [
    f"{value:+.1f}%"
    for value in change_rates
]


# 변화율 그래프는 두 막대 모두 파란색
change_colors = [
    BLUE,
    BLUE
]


fig2 = go.Figure()


fig2.add_trace(
    go.Bar(
        x=change_rates,
        y=metrics,
        orientation="h",

        marker_color=change_colors,

        # 막대 바깥쪽에 변화율 표시
        text=change_text,
        textposition="outside",

        textfont=dict(
            size=15,
            color="#333333"
        ),

        hovertemplate=(
            "%{y}<br>"
            "변화율: %{x:+.1f}%"
            "<extra></extra>"
        )
    )
)


# 변화율 그래프의 X축 범위 계산
max_change = max(abs(value) for value in change_rates)
min_change = min(0, min(change_rates))


# 그래프 2 레이아웃
fig2.update_layout(
    title=dict(
        text=(
            "코로나 전후 변화율 비교"
            "<br><sup>코로나 이전 대비 이후 변화율</sup>"
        ),

        x=0.5,

        # 제목과 부제목을 그래프 쪽으로 아래로 이동
        y=0.88,

        xanchor="center",
        yanchor="top",

        font=dict(size=22)
    ),

    height=390,

    # 상단 여백을 줄여 제목과 그래프 간격 축소
    margin=dict(
        t=115,
        b=65,
        l=110,
        r=110
    ),

    template="plotly_white",

    font=dict(
        family="Arial, sans-serif",
        size=14,
        color="#333333"
    ),

    xaxis=dict(
        title="변화율 (%)",
        ticksuffix="%",
        showgrid=True,
        gridcolor="#E8E8E8",
        zeroline=True,
        zerolinecolor="#777777",

        range=[
            min_change - max_change * 0.08,
            max_change * 1.25
        ]
    ),

    yaxis=dict(
        title="",
        autorange="reversed"
    ),

    showlegend=False
)


fig2.show()

In [19]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# ============================================================
# 1. 코로나 전후 비교 데이터
# ============================================================

periods = ["코로나 이전", "코로나 이후"]

# 입항 건수
arrival_values = [108, 201]

# 승객 수
passenger_values = [233991, 299274]

# 변화율 계산 (%)
arrival_change = (
    (arrival_values[1] - arrival_values[0])
    / arrival_values[0]
    * 100
)

passenger_change = (
    (passenger_values[1] - passenger_values[0])
    / passenger_values[0]
    * 100
)


# ============================================================
# 2. 색상 설정
# ============================================================

BLUE = "#3485D1"
ORANGE = "#F28E2B"

# 막대그래프에 적용할 색상만 변경
BAR_BLUE = "#3478C0"
BAR_ORANGE = "#E67E22"

# 코로나 이전: 파란색
# 코로나 이후: 오렌지색
period_colors = [BAR_BLUE, BAR_ORANGE]


# ============================================================
# 3. 그래프 생성
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "입항 건수 비교",
        "승객 수 비교"
    ),
    horizontal_spacing=0.16
)


# ============================================================
# 4. 왼쪽 그래프: 입항 건수
# ============================================================

fig.add_trace(
    go.Bar(
        x=periods,
        y=arrival_values,
        marker_color=period_colors,

        # 막대 내부 실제 수치
        text=[
            f"{value:,.0f}"
            for value in arrival_values
        ],
        textposition="inside",
        insidetextanchor="middle",

        textfont=dict(
            color="white",
            size=15
        ),

        name="입항 건수",

        hovertemplate=(
            "%{x}<br>"
            "입항 건수: %{y:,.0f}건"
            "<extra></extra>"
        )
    ),
    row=1,
    col=1
)


# ============================================================
# 5. 오른쪽 그래프: 승객 수
# ============================================================

fig.add_trace(
    go.Bar(
        x=periods,
        y=passenger_values,
        marker_color=period_colors,

        # 막대 내부 실제 수치
        text=[
            f"{value:,.0f}"
            for value in passenger_values
        ],
        textposition="inside",
        insidetextanchor="middle",

        textfont=dict(
            color="white",
            size=15
        ),

        name="승객 수",

        hovertemplate=(
            "%{x}<br>"
            "승객 수: %{y:,.0f}명"
            "<extra></extra>"
        )
    ),
    row=1,
    col=2
)


# ============================================================
# 6. 각 그래프에 핵심 비교 수치와 증가율 표시
# ============================================================

# 왼쪽: 입항 건수 증가율
fig.add_annotation(
    x=0.5,
    xref="x domain",

    y=arrival_values[1] * 1.19,
    yref="y",

    text=(
        f"<b>증가율 +{arrival_change:.1f}%</b>"
        f"<br>{arrival_values[0]:,}건 → "
        f"{arrival_values[1]:,}건"
    ),

    showarrow=False,

    font=dict(
        size=15,
        color=BLUE
    ),

    bgcolor="rgba(52, 133, 209, 0.10)",
    bordercolor=BLUE,
    borderwidth=1,
    borderpad=7,

    align="center"
)


# 오른쪽: 승객 수 증가율
fig.add_annotation(
    x=0.5,
    xref="x2 domain",

    y=passenger_values[1] * 1.19,
    yref="y2",

    text=(
        f"<b>증가율 +{passenger_change:.1f}%</b>"
        f"<br>{passenger_values[0]:,}명 → "
        f"{passenger_values[1]:,}명"
    ),

    showarrow=False,

    font=dict(
        size=15,
        color=BLUE
    ),

    bgcolor="rgba(52, 133, 209, 0.10)",
    bordercolor=BLUE,
    borderwidth=1,
    borderpad=7,

    align="center"
)


# ============================================================
# 7. 제목 및 레이아웃
# ============================================================

fig.update_layout(
    title=dict(
        text="코로나 전후 크루즈 입항·승객 규모 비교",
        x=0.5,
        y=0.95,
        xanchor="center",
        yanchor="top",
        font=dict(size=22)
    ),

    height=560,

    # 제목과 그래프 사이 간격 확보
    margin=dict(
        t=125,
        b=75,
        l=80,
        r=80
    ),

    template="plotly_white",

    showlegend=False,

    bargap=0.4,

    font=dict(
        family="Arial, sans-serif",
        size=13,
        color="#333333"
    )
)


# ============================================================
# 8. Y축 설정
# ============================================================

# 왼쪽 Y축: 입항 건수
fig.update_yaxes(
    title_text="입항 건수",
    tickformat=",",
    range=[0, max(arrival_values) * 1.50],
    showgrid=True,
    gridcolor="#E8E8E8",
    row=1,
    col=1
)


# 오른쪽 Y축: 승객 수
fig.update_yaxes(
    title_text="승객 수(명)",
    tickformat=",",
    range=[0, max(passenger_values) * 1.50],
    showgrid=True,
    gridcolor="#E8E8E8",
    row=1,
    col=2
)


# X축 설정
fig.update_xaxes(
    showgrid=False
)


# ============================================================
# 9. 그래프 출력
# ============================================================

fig.show()